# Flow Matching on $\mathrm{SO}(3)$
In this notebook, we perform flow matching on the rotation group $\mathrm{SO}(3)$, corresponding to Figure 2 in ["Flow Matching on Lie Groups"](https://arxiv.org/abs/2504.00494).

In [ ]:
import torch
import numpy as np
from lieflow.groups import SO3
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance
)
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "font.size": 10.0,
    "text.latex.preamble": r"\usepackage{lmodern} \usepackage{amssymb} \usepackage{amsmath}",
    "figure.constrained_layout.use": True,
})

device = "cpu"

In [ ]:
so3 = SO3(device=device)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def generate_line(N, d=0., w=torch.pi/4, horizontal=True, ε=0.05):
    if horizontal:
        d_ε = d + torch.randn(N, device=device) * ε
        cos_y = torch.cos(d_ε)
        sin_y = torch.sin(d_ε)
        R_y = torch.zeros(N, 3, 3, device=device)
        R_y[..., 0, 0] = cos_y
        R_y[..., 0, 2] = -sin_y
        R_y[..., 1, 1] = 1.
        R_y[..., 2, 0] = sin_y
        R_y[..., 2, 2] = cos_y
        
        θs_z = 2 * w * (torch.rand(N, device=device) - 0.5)
        cos_z = torch.cos(θs_z)
        sin_z = torch.sin(θs_z)
        R_z = torch.zeros(N, 3, 3, device=device)
        R_z[..., 0, 0] = cos_z
        R_z[..., 0, 1] = -sin_z
        R_z[..., 1, 0] = sin_z
        R_z[..., 1, 1] = cos_z
        R_z[..., 2, 2] = 1.

        Rs = R_z @ R_y
    else:
        turn_ε = -torch.pi/2 + torch.randn(N, device=device) * ε
        cos_turn = torch.cos(turn_ε)
        sin_turn = torch.sin(turn_ε)
        R_turn = torch.zeros(N, 3, 3, device=device)
        R_turn[..., 0, 0] = 1.
        R_turn[..., 1, 1] = cos_turn
        R_turn[..., 1, 2] = -sin_turn
        R_turn[..., 2, 1] = sin_turn
        R_turn[..., 2, 2] = cos_turn
        
        d_ε = d + torch.randn(N, device=device) * ε
        cos_z = torch.cos(d_ε)
        sin_z = torch.sin(d_ε)
        R_z = torch.zeros(N, 3, 3, device=device)
        R_z[..., 0, 0] = cos_z
        R_z[..., 0, 1] = -sin_z
        R_z[..., 1, 0] = sin_z
        R_z[..., 1, 1] = cos_z
        R_z[..., 2, 2] = 1.

        θs_y = 2 * w * (torch.rand(N, device=device) - 0.5)
        cos_y = torch.cos(θs_y)
        sin_y = torch.sin(θs_y)
        R_y = torch.zeros(N, 3, 3, device=device)
        R_y[..., 0, 0] = cos_y
        R_y[..., 0, 2] = -sin_y
        R_y[..., 1, 1] = 1.
        R_y[..., 2, 0] = sin_y
        R_y[..., 2, 2] = cos_y

        Rs = R_z @ R_y @ R_turn
    return Rs

def generate_circle(N, d=0., w=torch.pi/4, ε=0.05, gap=0.):
    w_ε = w + torch.randn(N, device=device) * ε
    cos_y = torch.cos(w_ε)
    sin_y = torch.sin(w_ε)
    R_y = torch.zeros(N, 3, 3, device=device)
    R_y[..., 0, 0] = cos_y
    R_y[..., 0, 2] = -sin_y
    R_y[..., 1, 1] = 1.
    R_y[..., 2, 0] = sin_y
    R_y[..., 2, 2] = cos_y

    θs_x = (torch.rand(N, device=device) * (2 - gap) - (1.5 - gap)) * torch.pi
    cos_x = torch.cos(θs_x)
    sin_x = torch.sin(θs_x)
    R_x = torch.zeros(N, 3, 3, device=device)
    R_x[..., 0, 0] = 1.
    R_x[..., 1, 1] = cos_x
    R_x[..., 1, 2] = -sin_x
    R_x[..., 2, 1] = sin_x
    R_x[..., 2, 2] = cos_x
    
    d_ε = d + torch.randn(N, device=device) * ε
    cos_z = torch.cos(d_ε)
    sin_z = torch.sin(d_ε)
    R_z = torch.zeros(N, 3, 3, device=device)
    R_z[..., 0, 0] = cos_z
    R_z[..., 0, 1] = -sin_z
    R_z[..., 1, 0] = sin_z
    R_z[..., 1, 1] = cos_z
    R_z[..., 2, 2] = 1.
    
    return R_z @ R_x @ R_y

q_ref = torch.tensor([
    [1., 0., 0.],
    [0., 0., 1.]
    ]).T

@torch.no_grad()
def plot_R(ax, R, **kwargs):
    R = R.to("cpu")
    q = R @ q_ref
    r = 1.01
    return ax.quiver(
        r*q[..., 0, 0], r*q[..., 1, 0], r*q[..., 2, 0],
        q[..., 0, 1], q[:32, 1, 1], q[..., 2, 1],
        **kwargs
    )

In [ ]:
tests = ("horizontal_line_to_vertical_line", "vertical_line_to_circle")

In [ ]:
def data_generator(test, ε=0.04):
    match test:
        case "horizontal_line_to_vertical_line":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=True, ε=ε)
            generate_R_1 = partial(generate_line, d=torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
        case "vertical_line_to_vertical_line":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
            generate_R_1 = partial(generate_line, d=torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
        case "vertical_line_to_circle":
            generate_R_0 = partial(generate_line, d=-torch.pi/6, w=torch.pi/12, horizontal=False, ε=ε)
            generate_R_1 = partial(generate_circle, d=torch.pi/6, w=torch.pi/12, gap=0., ε=ε)
        case "circle_to_circle":
            generate_R_0 = partial(generate_circle, w=torch.pi/20, ε=ε)
            generate_R_1 = partial(generate_circle, w=torch.pi/6, ε=ε)
    return generate_R_0, generate_R_1

# Models

## Training

In [ ]:
EPSILON = 0.01
N = 2**14
BATCH_SIZE = 2**10
EPOCHS = len(tests) * [50]
WEIGHT_DECAY = 0.
LEARNING_RATE = 1e-2
H = 64 # Width
L = 3 # Number of layers is L + 2

In [ ]:
def train_model(R_0s, R_1s, epochs, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY):
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(R_0s, R_1s), batch_size=batch_size, shuffle=True
    )

    model_FM = get_model_FM(so3, H=H, L=L).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), learning_rate, weight_decay=weight_decay)
    loss = LogarithmicDistance(torch.tensor([1., 1., 1.], device=device))

    losses_FM = np.zeros(epochs)
    for i in tqdm(range(epochs)):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, loss, show_progress=False)

    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.plot(losses_FM)
    ax.set_title("Batch Loss Flow Matching")
    ax.set_xscale("log")
    ax.set_yscale("log")

    return model_FM

In [ ]:
theta = torch.linspace(0, torch.pi, 50)
phi = torch.linspace(0, 2 * torch.pi, 100)
theta, phi = torch.meshgrid(theta, phi)

x = torch.sin(theta) * torch.cos(phi)
y = torch.sin(theta) * torch.sin(phi)
z = torch.cos(theta)

models_FM = {}
for i, test in enumerate(tests):
    print(test)
    generate_R_0, generate_R_1 = data_generator(test, ε=EPSILON)

    R_0s = generate_R_0(N)
    R_1s = generate_R_1(N)

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection='3d')
    ax.view_init(elev=15, azim=15)
    ax.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)
    plot_R(ax, R_0s[:32], length=0.1, color="blue")
    plot_R(ax, R_1s[:32], length=0.1, color="green")

    ax.set_xlim([-1, 1])
    ax.set_xticks([])
    ax.set_ylim([-1, 1])
    ax.set_yticks([])
    ax.set_zlim([-1, 1])
    ax.set_zticks([])
    ax.set_aspect("equal")

    models_FM[test] = train_model(R_0s, R_1s, epochs=EPOCHS[i], batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)

## Testing

In [ ]:
def create_animations(tests, models_FM, N=2**5):
    # Forward flow
    R_0s_test = {}
    R_ts_test = {}
    for test in tests:
        generate_R_0, _ = data_generator(test, ε=EPSILON)
        R_0s_test[test] = generate_R_0(N).to(device)
        R_ts_test[test] = R_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    t = 0
    N_steps = 120
    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': f'Flow Matching SO(3) {test}', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig = plt.figure(figsize=(8, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.view_init(elev=15, azim=15)
        ax.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)
        quiver = ax.quiver([], [], [], [], [], [])

        ax.set_xlim([-1, 1])
        ax.set_xticks([])
        ax.set_ylim([-1, 1])
        ax.set_yticks([])
        ax.set_zlim([-1, 1])
        ax.set_zticks([])
        ax.set_aspect("equal")
        with writer.saving(fig, f"output/flow_matching_SO3_{test}.mp4", dpi=150):

            quiver.remove()
            quiver = plot_R(ax, R_ts_test[test], length=0.25)
        
            writer.grab_frame()
            for frame in tqdm(range(N_steps)):
                t = frame * Δt
                with torch.no_grad():
                    R_ts_test[test] = models_FM[test].step(R_ts_test[test], torch.tensor([t], device=device)[..., None], Δt)

                quiver.remove()
                quiver = plot_R(ax, R_ts_test[test], length=0.25)
                writer.grab_frame()
            writer.grab_frame()

In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N=2**5)

In [ ]:
t = 0
N_steps = 240
Δt = 1. / N_steps
N_show = 5
N_skip = int(N_steps / (N_show-1))

N_models = len(tests)
N_samples = 32

R_0s_test = {}
R_1s_test = {}
R_ts_test = {}
for test in tests:
    generate_R_0, generate_R_1 = data_generator(test, ε=EPSILON)
    R_0s_test[test] = generate_R_0(N_samples).to(device)
    R_1s_test[test] = generate_R_1(N_samples).to(device)
    R_ts_test[test] = R_0s_test[test].detach().clone()
    
    models_FM[test].eval()

fig = plt.figure(figsize=(4.8, 1.6 * N_models * 3/3.1))
gs = gridspec.GridSpec(N_models, 4, width_ratios=[1, 1, 1, 0.1], height_ratios=N_models * [1.], wspace=0.1, hspace=0.1)
cax = fig.add_subplot(gs[:, 3])

ax = []
for i in range(N_models):
    ax.append([])
    for j in range(3):
        a = fig.add_subplot(gs[i, j], projection="3d")
        a.view_init(elev=15, azim=15)
        a.plot_surface(x, y, z, color='cyan', alpha=0.25, edgecolor=None)

        a.set_xlim([-1, 1])
        a.set_xticks([])
        a.set_ylim([-1, 1])
        a.set_yticks([])
        a.set_zlim([-1, 1])
        a.set_zticks([])
        a.set_aspect("equal")
        ax[i].append(a)
        
ax[0][0].set_title(r"$\mathfrak{X}_0$")
ax[0][1].set_title(r"$\mathfrak{X}_t$")
ax[0][2].set_title(r"$\mathfrak{X}_1$")
Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)
for i, test in enumerate(tests):
    k = 0

    plot_R(ax[i][0], R_0s_test[test], length=0.1, linewidths=0.05)
    plot_R(ax[i][2], R_1s_test[test], length=0.1, linewidths=0.05)

    alpha = 1
    N_samples_shown = N_samples
    for j in range(N_steps+1):
        t = j * Δt
        if j == N_steps:
            alpha = 1
            N_samples_shown = N_samples
        with torch.no_grad():
            R_ts_test[test] = models_FM[test].step(R_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:

            plot_R(ax[i][1], R_ts_test[test], color=colors[k], length=0.1, linewidths=0.05, alpha=alpha)
            k += 1
        alpha = 0.5
        N_samples_shown = N_samples // 4
fig.colorbar(ScalarMappable(cmap=cmap), cax=cax, ticks=np.linspace(0, 1, N_show), label="$t$");
fig.savefig("output/interpolation_SO3.pdf", bbox_inches="tight")